In [ ]:
!pip install -q trl peft bitsandbytes lm-eval

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# HuggingFace Hub model identifier 
model_name = "allenai/OLMo-2-0425-1B"

# Create Tokenizer and load the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Since, OLMO-2-1B does not have its own chat template, we borrow the one present in OLMo-2-1B-Instruct
instruct_tokenizer = AutoTokenizer.from_pretrained("allenai/OLMo-2-0425-1B-Instruct")
tokenizer.chat_template = instruct_tokenizer.chat_template

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from datasets import load_dataset
import json

# Load the full Alpaca dataset
dataset = load_dataset("tatsu-lab/alpaca")["train"]

# Load training dataset
with open("/kaggle/input/datasets/hkcc375/random-baseline-at-10-budget-for-alpaca/random_10pct.json") as f:
    selection = json.load(f)

# Create training dataset
subset = dataset.select(selection["selected_samples"])
print(f"Method: {selection["method"]}")
print(f"Selected: {len(subset)} examples")

In [ ]:
# Setting PAD token for Tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Format examples into exact text format OLMo 2 expects

# Note : <|user|>, <|assistant|> and <|endoftext|> are special tokens in OLMo's vocabulary

# <|user|>
# What is photosynthesis ?
# <|assistant|>
# Photosynthesis is the process by which plants convert sunlight...
# <|endoftext|>

def format_example(example):
    if example['input'].strip():
        prompt = f"{example['instruction']}\n\n{example['input']}"
    else:
        prompt = example['instruction']

    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": example['output']}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    return {"text": text}

training_dataset = subset.map(format_example)

In [ ]:
# Checking how the formatted example looks
print("Formatted example : ")
print(training_dataset[0]["text"])

In [ ]:
# Training with LoRA

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

output_dir = "/kaggle/working/olmo2_random_10pct"

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=25,
    save_strategy="epoch",
    max_length=512,
    dataset_text_field="text",
    report_to="none",  # disable wandb on Kaggle
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=training_dataset,
    peft_config=lora_config,
)

print("Starting training...")
trainer.train()

# Save the final model
trainer.save_model(f"{output_dir}/final")
tokenizer.save_pretrained(f"{output_dir}/final")
print(f"Model saved to {output_dir}/final")